# **Whoscored**

### Récupération des notes des joueurs.

Ce notebook est fait pour récupérer les notes des joueurs pendant les matches.

Pour cela, on utilise le site **Whoscored** : `https://fr.whoscored.com`.

Le notebook se présente en 2 parties :
- **Historique** : Pour récupérer les liens et les matches des années précédentes
- **Actualisation** : Pour récupérer les liens et les notes des joueurs des derniers matches (mettre à jour la table)

La logique du scrapping est la suivante :
- Si on récupère la base de données pour la première fois :
    1) On récupère les liens des championnats majeurs (si on veut en rajouter, il faut les rajouter à la main) avec la fonction `get_link_top_leagues`.
    2) On récupère les liens de toutes les saisons disponibles pour les championnats majeurs que l'on a définit avec la fonction `get_link_historical_leagues`.
    3) On récupère les liens de tous les matches, de toutes les saisons disponibles pour les championnats majeurs que l'on a définit avec la fonction `get_link_match`.
    4) On récupère les notes des joueurs avec la fonction `get_rates_database`.
- Si on veut actualiser la base de données :
    1) On utilise la fonction `update_rates_database()`

In [1]:
# Importation des packages
import ace_tools_open as tools
import pandas as pd
from whoscored_links_functions import get_link_top_leagues, get_link_historical_leagues, get_link_match, update_match_link
from whoscored_score_functions import update_rates_database

# I. Historique



### A. Liens des championnats majeurs

L'objectif ici est de récupérer les liens des championnats majeurs disponibles sur le site **Whoscored**.
Pour cela, on utilise la fonction `get_link_top_leagues()` qui va nous retourner un dataframe avec le nom de la ligue et le liens qui mène aux saisons pour chaque ligue.

In [2]:
# data_link_top_leagues = get_link_top_leagues(save=True)
data_link_top_leagues = pd.read_csv("urls/data_link_top_leagues.csv")
tools.display_dataframe_to_user("", data_link_top_leagues)

### B. Liens des saisons précédentes pour les championnats majeurs

Ici, on veut racupérer les liens de toutes les saisons de chaque championnat majeurs. On utilise la fonction `get_link_historical_leagues()`. En entrée, on donne le dataframe avec les liens des championnats majeurs et en sortie on va avoir un dataframe avec les liens de chaque championnat pour toutes les années disponibles.

In [3]:
#data_link_leagues = get_link_historical_leagues(data_link_top_leagues[:5], save=False)
data_link_leagues = pd.read_csv("urls/data_link_leagues.csv")
tools.display_dataframe_to_user("", data_link_leagues)

### C. Récupération des liens pour les matchs

In [4]:
# data_link_matches = get_link_match(data_link_leagues)
data_link_matches = pd.read_csv("urls/data_link_matches.csv")
tools.display_dataframe_to_user("", data_link_matches)

### D. Récupération des notes des joueurs

In [2]:
#links = pd.read_csv("urls/data_link_matches.csv")
#links = links[links["season"] == "2009/2010"]
#rates = get_rates_database(links, save=True, add=True)
rates = pd.read_csv("rates/rates.csv")
tools.display_dataframe_to_user("", rates)

# II. Update

In [3]:
new_rates = update_rates_database(save=True)
tools.display_dataframe_to_user("", new_rates)

Update des liens :


100%|██████████| 5/5 [00:21<00:00,  4.36s/it]


Aucun autre lien de match trouvé



# III. Récupération des statistiques pour un match

In [50]:
import re
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from webdriver_manager.firefox import GeckoDriverManager
from selenium.webdriver.firefox.options import Options
from bs4 import BeautifulSoup
import time
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from urllib.parse import urlparse

In [4]:
rates = pd.read_csv("rates/rates.csv")
links = list(rates["link"])

In [6]:
options = Options()
#options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")

service = Service(GeckoDriverManager().install())
driver = webdriver.Firefox(service=service, options=options)

In [7]:
link = links[0]
driver.get(link)
link

'https://fr.whoscored.com/matches/1901385/live/italie-serie-a-2025-2026-udinese-fiorentina'

In [8]:
bouton = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, "//button[text()='Tout accepter']"))
)

bouton.click()

In [9]:
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

In [ ]:
stats_team = []
code = soup.find_all("li", class_="match-centre-stat match-centre-sub-stat")
for c in code:
    stats_team.append(c.text)

rows = []
for s in stats_team[:31]:
    m = re.match(r'(.+?)(\d+)-\s*(\d+)$', s)
    if m:
        name = m.group(1).strip()
        stat1 = int(m.group(2))
        stat2 = int(m.group(3))
        rows.append([name, stat1, stat2])

df = pd.DataFrame(rows, columns=["nom", "stat1", "stat2"])
df["nom"] = df["nom"].str.lower().str.replace(" ", "_")

driver.find_element(By.LINK_TEXT, "Tableau Noir").click()
time.sleep(1)
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

stats_team_2 = []
code = soup.find_all("li", class_="filterz-option")
for c in code:
    stats_team_2.append(c.text)

rows_2 = []
for s in stats_team_2:
    m = re.match(r'(.+?)(\d+)\s*-\s*(\d+)$', s)
    if m:
        name = m.group(1).strip()
        stat1 = int(m.group(2))
        stat2 = int(m.group(3))
        rows_2.append([name, stat1, stat2])

df_2 = pd.DataFrame(rows_2, columns=["nom", "stat1", "stat2"])
df_2["nom"] = df_2["nom"].str.lower().str.replace(" ", "_")

stats_team_3 = []
for i in soup.find_all("div", class_="filterz-filter"):
    if i.text != "Toutes":
        stats_team_3.append(i.text)

rows_3 = []
for s in stats_team_3:
    m = re.match(r'^(\d+)(.+?)(\d+)$', s)
    if m:
        stat1 = int(m.group(1))
        name = m.group(2).strip()
        stat2 = int(m.group(3))
        if name == "mètres":
            name = "6 mètres"
            stat1 = re.sub(r'6$', '', str(stat1))
        rows_3.append([name, stat1, stat2])

df_3 = pd.DataFrame(rows_3, columns=["nom", "stat1", "stat2"])
df_3["Categorie"] = ["tirs"] * 18 + ["passes"] * 19 + ["dribbles"] * 2 + ["tacles tentés"] * 2 + ["dégagements"] * 4 + ["contres"] * 2 + ["perte de balle"] * 2 + ["erreur"] * 2
df_3["nom"] = df_3["nom"].str.lower().str.replace(" ", "_")

for i in range(len(df_3)):
    if df_3.loc[i, "Categorie"] not in df_3.loc[i, "nom"]:
        df_3.loc[i,"nom"] = df_3.loc[i,"Categorie"] + "_" + df_3.loc[i, "nom"]

df_3 = df_3[["nom", "stat1", "stat2"]].drop_duplicates()

df = pd.concat([df, df_2, df_3])

globale = df[df["nom"].isin(["tirs_buts", "possession%"])].drop_duplicates()
globale["category"] = "global"

tirs = df[(df["nom"].str.contains("tirs")) & (~df["nom"].str.contains("contres"))].drop_duplicates()
tirs["category"] = "tirs"

passes = df[(df["nom"].str.contains("passes")) & (~df["nom"].str.contains("corner"))].drop_duplicates(subset=["stat1", "stat2"])
passes["category"] = "passes"

dribbles = df[df["nom"].str.contains("dribbles")].drop_duplicates()
dribbles["category"] = "dribbles"

tacles = df[df["nom"].str.contains("tacles")].drop_duplicates()
tacles["category"] = "tacles"

interceptions = df[df["nom"].str.contains("interceptions")].drop_duplicates()
interceptions["category"] = "interceptions"

dégagements = df[df["nom"].str.contains("dégagements")].drop_duplicates()
dégagements["category"] = "dégagements"

contres = df[df["nom"].str.contains("contres")].drop_duplicates()
contres["category"] = "contres"

hors_jeux = df[df["nom"].str.contains("hors-jeux")].drop_duplicates()
hors_jeux["category"] = "hors_jeux"

fautes = df[df["nom"].str.contains("fautes")].drop_duplicates()
fautes["category"] = "fautes"

duels_aériens = df[df["nom"].str.contains("aériens")].drop_duplicates()
duels_aériens["category"] = "duels_aériens"

touches = df[df["nom"].str.contains("touchés")].drop_duplicates()
touches["category"] = "touches"

perte_de_balle = df[df["nom"].str.contains("perte")].drop_duplicates()
perte_de_balle["category"] = "perte_de_balle"

erreurs = df[df["nom"].str.contains("erreur")].drop_duplicates()
erreurs["category"] = "erreurs"

corner = df[(df["nom"].str.contains("corner")) & (~df["nom"].str.contains("passes"))].drop_duplicates()
corner["category"] = "corner"

gardien = df[df["nom"].isin(["arrêts", "saisis", "boxés"])].drop_duplicates()
gardien["category"] = "gardien"

df = pd.concat([globale, tirs, passes, dribbles, tacles, interceptions, dégagements, contres, hors_jeux, fautes, duels_aériens, touches, perte_de_balle, erreurs, corner, gardien])
df = df.drop_duplicates(subset=["nom", "stat1", "stat2"])
df_pivot = pd.concat(
    [
        df.set_index("nom")["stat1"].rename(lambda x: f"{x}_1"),
        df.set_index("nom")["stat2"].rename(lambda x: f"{x}_2")
    ]
).to_frame().T

order = [f"{nom}_{i}" for nom in df["nom"] for i in (1, 2)]
df_pivot = df_pivot[order]

tools.display_dataframe_to_user("", df_pivot)

In [55]:
soup.find_all("div", class_="filterz-filter")

[<div class="filterz-filter selected" data-filter-index="all"><label>Toutes</label></div>,
 <div class="filterz-filter" data-filter-index="0_0_0" data-sum="3"><span class="filterz-value greater" data-field="home" data-value="3">3</span><label>Buts</label><span class="filterz-value" data-field="away" data-value="0">0</span></div>,
 <div class="filterz-filter" data-filter-index="0_0_1" data-sum="7"><span class="filterz-value greater" data-field="home" data-value="7">7</span><label>Tirs Cadrés</label><span class="filterz-value" data-field="away" data-value="0">0</span></div>,
 <div class="filterz-filter" data-filter-index="0_0_2" data-sum="7"><span class="filterz-value" data-field="home" data-value="1">1</span><label>Tirs non Cadrés</label><span class="filterz-value greater" data-field="away" data-value="6">6</span></div>,
 <div class="filterz-filter" data-filter-index="0_0_3" data-sum="0"><span class="filterz-value" data-field="home" data-value="0">0</span><label>Montants</label><span cl

In [ ]:
driver.find_element(By.LINK_TEXT, "Buts")

AttributeError: 'WebDriver' object has no attribute 'find_all'

In [ ]:
filterz-filter

In [94]:
driver.quit()